In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip uninstall -y faiss faiss-cpu
!pip install faiss-cpu

Found existing installation: faiss-cpu 1.8.0.post1
Uninstalling faiss-cpu-1.8.0.post1:
  Successfully uninstalled faiss-cpu-1.8.0.post1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.7 MB/s eta 0:00:00


In [2]:
!pip install -U sentence-transformers

In [3]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# =====================================================
# CONFIGURATION
# =====================================================

CSV_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/dataset/candidate_profile.csv"

# Text column to vectorize
TEXT_COLUMN = "summary"

# Embedding model
MODEL_NAME = "all-MiniLM-L6-v2"

# Output files
FAISS_INDEX_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummaryvector_db.faiss"
METADATA_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummarymetadata.pkl"

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv(CSV_FILE)

print("Rows:", len(df))

# Remove nulls
df = df[df[TEXT_COLUMN].notna()].copy()

texts = df[TEXT_COLUMN].astype(str).tolist()

print("Documents:", len(texts))

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# CREATE EMBEDDINGS
# =====================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(
    embeddings,
    dtype=np.float32
)

print("Embedding Shape:", embeddings.shape)

# =====================================================
# CREATE FAISS INDEX
# =====================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

# =====================================================
# SAVE INDEX
# =====================================================

faiss.write_index(
    index,
    FAISS_INDEX_FILE
)

print("FAISS index saved")

# =====================================================
# SAVE METADATA
# =====================================================

metadata = df.to_dict("records")

with open(
    METADATA_FILE,
    "wb"
) as f:
    pickle.dump(metadata, f)

print("Metadata saved")

print("Done!")

Rows: 100000
Documents: 100000
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Embedding Shape: (100000, 384)
Vectors Stored: 100000
FAISS index saved
Metadata saved
Done!
